## Data Processing

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import datetime as dt
import matplotlib.pyplot as plt
import yfinance as yf
import torch.nn as nn

In [2]:
# asset price loading
# dropping Null vals
# , 'TLT', 'XLF', 'XLE', 'XLU', 'XLK', 'EXW1.DE', '^FTSE','1329.T', 'QQQ'
symbols = yf.download(['SPY'],start=dt.datetime(2009,1,1), end=dt.datetime(2016,12,31), auto_adjust=False, multi_level_index=False, progress=False).dropna()

closePs = symbols['Adj Close'].to_frame()

type(closePs)
np.shape(closePs)

(2014, 1)

In [3]:
# calculate slopeReference (slopeRef) and find the distribution of the labels
# find the first,second seperation points..

arr = []

for r in range(len(closePs) - 45):
    all_y = closePs.values.tolist()
    price45 = all_y[r + 33][0]
    price30 = all_y[r + 29][0]
    dif_ratio = ((price45 - price30) / price30) * 100
    arr.append(dif_ratio)

arr.sort()
length = len(arr)

f_sliding = round(arr.__getitem__(2*int(length/5)),2)
s_sliding =  round(arr.__getitem__(3*int(length/5)),2)

print("len_roc:",length, "f_sliding:",f_sliding,"s_sliding:",s_sliding)

len_roc: 1969 f_sliding: -0.05 s_sliding: 0.75


In [4]:
# Need to determine price transformation to images for analysis
# I guess this is where the indicators come in handy
# Will convert snapshots to images using @Omer Berat Sezer's approach

for r in range(len(closePs) - 45):
        print('r:', r)
        img = [[]]

        # all pixels are set value 255 (white) in 30x30 pixel image
        img = [[255 for i in range(30)] for i in range(30)]
        all_y = closePs.values.tolist()
        sub_y = all_y[r:r + 30]

        current_price = round(all_y[r+29][0], 2)

        price45 = all_y[r + 44][0]
        price30 = all_y[r + 29][0]

        # calculate ratio
        dif_ratio = ((price45 - price30) / price30) * 100

        max_y = (all_y[r+15][0] * 130) / 100
        min_y = (all_y[r+15][0] * 70) / 100

        label_avg_y = 0
        label_sum_y = 0

        print("f_sliding,s_sliding:",f_sliding,s_sliding)
        #test1
        if (dif_ratio >= s_sliding): # slopeCurrent>slopeRef[s_sliding]
            predictLabel = 1   # label=Buy
        elif (dif_ratio > f_sliding and dif_ratio < s_sliding):
            predictLabel = 0   # label = Hold
        elif (dif_ratio <= f_sliding): # slopeCurrent<slopeRef[s_sliding]
            predictLabel = 2   # label = Sell

        print("predictLabel:", predictLabel, " dif_ratio:", dif_ratio, "price:", current_price)
        print("max:",max_y, "min:",min_y)

        # calculate coefficient to normalize data
        coef = 30 / (int(max_y - min_y)+1)
        j = 0
        print(max_y, min_y)

        # calculate the stock price and create black bar graphics for 30 days
        for i in range(30):
            val = (sub_y[i][0] - min_y) * coef
            #print(val)
            for k in range(int(val)):
                if(k<30):
                    img[29 - k][j] = 0
            j += 1

        my_df = pd.DataFrame(img)
        label_price = ';'.join((str(predictLabel), str(current_price)))

        # append image values in file
        my_df.to_csv('data/csvIms.csv', index=False, header=False, mode='a', lineterminator=';', sep=',')
        with open('data/csvIms.csv', 'a') as file:
            file.write(label_price)
            file.write('\n')
        del label_sum_y, predictLabel

        # print image
        # print(img)
        # DONT RUN WITH THESE ON EVER
        # imgplot = plt.imshow(img, cmap=plt.get_cmap('gray'))
        # plt.show()



r: 0
f_sliding,s_sliding: -0.05 0.75
predictLabel: 2  dif_ratio: -17.70176772175228 price: 60.37
max: 79.34888191223145 min: 42.72632102966308
79.34888191223145 42.72632102966308
r: 1
f_sliding,s_sliding: -0.05 0.75
predictLabel: 2  dif_ratio: -8.899266904993402 price: 57.78
max: 80.15490531921387 min: 43.16033363342285
80.15490531921387 43.16033363342285
r: 2
f_sliding,s_sliding: -0.05 0.75
predictLabel: 2  dif_ratio: -8.08556684650975 price: 57.65
max: 82.86683731079101 min: 44.62060470581055
82.86683731079101 44.62060470581055
r: 3
f_sliding,s_sliding: -0.05 0.75
predictLabel: 2  dif_ratio: -3.42798799623225 price: 57.03
max: 80.17384910583496 min: 43.17053413391113
80.17384910583496 43.17053413391113
r: 4
f_sliding,s_sliding: -0.05 0.75
predictLabel: 2  dif_ratio: -1.71788725874276 price: 56.47
max: 78.54284362792968 min: 42.292300415039065
78.54284362792968 42.292300415039065
r: 5
f_sliding,s_sliding: -0.05 0.75
predictLabel: 1  dif_ratio: 1.6208563291097562 price: 54.45
max: 78.3

## Optimizer from scratch

In [ ]:
# Sharpness aware minimization based optim, it's better than SDG w/ momentum
# Need to determine build

class myOptimizer():

    def __init__(self, learning_rate, momentum):

        self.learning_rate = learning_rate
        self.momentum = momentum

        self.w = 0
        self.b = 0

        self.momentum_vector_w = 0
        self.momentum_vector_b = 0


    def _get_batch(self):

    def _get_momentum_vector(self, X_batch, y_batch):

    def firstS(self):

    def secondS(self):

    def fit(self, X, y, batch_size = 32, epochs = 100):

    def predict(self, X):
	    return self.w * X + self.b

In [8]:
#Code copied from personal repro
from tqdm import tqdm
from src.sam import SAM

def train_epoch_sam(
    model,
    loader,
    criterion,
    optimizer: SAM,
    device,
    use_amp: bool,
):

    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for x, y in tqdm(loader, desc="train", leave=False):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        if use_amp:
            with amp.autocast("cuda"):
                logits = model(x)
                loss = criterion(logits, y)
            loss.backward()
            optimizer.first_step(zero_grad=True)

            with amp.autocast("cuda"):
                logits2 = model(x)
                loss2 = criterion(logits2, y)
            loss2.backward()
            optimizer.second_step(zero_grad=True)
        else:
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.first_step(zero_grad=True)

            logits2 = model(x)
            loss2 = criterion(logits2, y)
            loss2.backward()
            optimizer.second_step(zero_grad=True)

        total_loss += loss.detach().float().item() * y.size(0)
        correct += (logits2.argmax(dim=1) == y).sum().item()
        total += y.size(0)
    return total_loss / max(total, 1), correct / max(total, 1)

ModuleNotFoundError: No module named 'src'

## Model, training

In [5]:
import torch
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cuda:0


In [11]:
from tqdm import tqdm
import torchvision.models as models

base_optimizer = torch.optim.SGD

models.list_models()

['alexnet',
 'convnext_base',
 'convnext_large',
 'convnext_small',
 'convnext_tiny',
 'deeplabv3_mobilenet_v3_large',
 'deeplabv3_resnet101',
 'deeplabv3_resnet50',
 'densenet121',
 'densenet161',
 'densenet169',
 'densenet201',
 'efficientnet_b0',
 'efficientnet_b1',
 'efficientnet_b2',
 'efficientnet_b3',
 'efficientnet_b4',
 'efficientnet_b5',
 'efficientnet_b6',
 'efficientnet_b7',
 'efficientnet_v2_l',
 'efficientnet_v2_m',
 'efficientnet_v2_s',
 'fasterrcnn_mobilenet_v3_large_320_fpn',
 'fasterrcnn_mobilenet_v3_large_fpn',
 'fasterrcnn_resnet50_fpn',
 'fasterrcnn_resnet50_fpn_v2',
 'fcn_resnet101',
 'fcn_resnet50',
 'fcos_resnet50_fpn',
 'googlenet',
 'inception_v3',
 'keypointrcnn_resnet50_fpn',
 'lraspp_mobilenet_v3_large',
 'maskrcnn_resnet50_fpn',
 'maskrcnn_resnet50_fpn_v2',
 'maxvit_t',
 'mc3_18',
 'mnasnet0_5',
 'mnasnet0_75',
 'mnasnet1_0',
 'mnasnet1_3',
 'mobilenet_v2',
 'mobilenet_v3_large',
 'mobilenet_v3_small',
 'mvit_v1_b',
 'mvit_v2_s',
 'quantized_googlenet',
 '

In [9]:
epochs = 50

for e in range(epochs) :

    print ('Epoch {}/{} '.format(e + 1, epochs))

    for batch in closePs.train:

        ## SAM requires 2 forward-backward steps ~

        # First forward-backward step
        myOptimizer.firstS()
        # Second forward-backward step
        myOptimizer.secondS()

Epoch 1/50 


AttributeError: 'Series' object has no attribute 'train'